In [ ]:
from pathlib import Path
import os
import sys
import json

import torch
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent
sys.path.append(str(ROOT))
from src.common.nuscenes_utils import (
    get_scene_contents,
    get_sample_contents,
    get_sample_window_contents
)
from src.common.geometry.pointcloud import (
    transform_lidar_to_ego,
    transform_ego_to_global
)
from src.common.geometry.depth import transform_cam_to_ego
from src.common.geometry.transform import make_transform, invert_transform
from src.common.visualize.pointcloud import plot_pointcloud
from src.common.visualize.depth import plot_depth_with_original_image, plot_pseudo_lidar_with_ground_truth
from src.common.frame_ops import create_sliding_windows

# Load nuScenes dataset
NUSCENES_ROOT = Path.cwd().parent / "data/nuscenes"
NUSCENES_VERSION = "v1.0-trainval"
with open(NUSCENES_ROOT / NUSCENES_VERSION / "scene.json") as f:
    scenes = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "sample.json") as f:
    samples_all = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "sample_data.json") as f:
    sample_data_all = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "ego_pose.json") as f:
    ego_poses_all = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "calibrated_sensor.json") as f:
    calibrated_sensors_all = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "sensor.json") as f:
    sensors = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "sample_annotation.json") as f:
    sample_annotations_all = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "instance.json") as f:
    instances_all = json.load(f)

# Create hash maps for token lookup
sensors = {sensor["token"]: sensor for sensor in sensors}
sensor_lookup = {sensor["channel"]: sensor["token"] for sensor in sensors.values()}

print(f"sensor_channels: {[sensor['channel'] for sensor in sensors.values()]}")
print(f"scene_names: {[scene['name'] for scene in scenes]}")